# FINAL — Vietnamese Traffic Sign Detection
**Traffic-sign detection only. Labels are short English names.**

Run this notebook **top-to-bottom** on Google Colab. Before running, choose **Runtime → Change runtime type → T4 GPU**.

Input: `MyDrive/DIP/video1.mp4`  
Full result: `MyDrive/DIP/outputs/video1_result.mp4`

The last cell also shows a lightweight preview directly inside Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/DIP
!git clone -q -b feature/yolo-traffic-safety https://github.com/NVTruong473/DIP.git /content/DIP
%cd /content/DIP/END_DIP
!pip install -q -r requirements.txt
!git log -1 --oneline


In [ ]:
from pathlib import Path
import shutil
import subprocess
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/DIP')
VIDEO_NAME = 'video1.mp4'  # Change only this line to test another Drive video.

VIDEO = DRIVE_ROOT / VIDEO_NAME
MODELS_DIR = DRIVE_ROOT / 'models'
OUTPUT_DIR = DRIVE_ROOT / 'outputs'
STEM = VIDEO.stem

OUTPUT_VIDEO = OUTPUT_DIR / f'{STEM}_result.mp4'
OUTPUT_CSV = OUTPUT_DIR / f'{STEM}_result.csv'
TEMP_VIDEO = OUTPUT_DIR / f'{STEM}_result_temp.mp4'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('Video:', VIDEO)
assert torch.cuda.is_available(), 'Enable a T4 GPU in Runtime → Change runtime type.'
assert VIDEO.exists(), f'Input video not found: {VIDEO}'


In [ ]:
# Download/cache only the traffic-sign model.
subprocess.run([
    'python', 'download_models.py',
    '--models-dir', str(MODELS_DIR),
], check=True)

# Remove stale outputs so the preview can never show an older result.
for path in (OUTPUT_VIDEO, OUTPUT_CSV, TEMP_VIDEO):
    try:
        path.unlink()
    except FileNotFoundError:
        pass

cmd = [
    'python', 'main.py',
    '--input', str(VIDEO),
    '--output-dir', str(OUTPUT_DIR),
    '--models-dir', str(MODELS_DIR),
    '--sign-conf', '0.25',
    '--sign-imgsz', '640',
    '--frame-stride', '1',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Verify the full result saved on Google Drive.
assert OUTPUT_VIDEO.exists() and OUTPUT_VIDEO.stat().st_size > 0, f'Output not created: {OUTPUT_VIDEO}'
assert OUTPUT_CSV.exists(), f'CSV not created: {OUTPUT_CSV}'

print(f'Full video: {OUTPUT_VIDEO}')
print(f'Size      : {OUTPUT_VIDEO.stat().st_size / 1024 / 1024:.1f} MB')
print(f'CSV       : {OUTPUT_CSV}')
print('\nCodec:')
subprocess.run([
    'ffprobe', '-v', 'error',
    '-select_streams', 'v:0',
    '-show_entries', 'stream=codec_name,pix_fmt,width,height',
    '-of', 'default=noprint_wrappers=1',
    str(OUTPUT_VIDEO),
], check=True)


In [ ]:
# Create a small H.264 preview and show it directly below this cell.
from IPython.display import Video, display

PREVIEW = Path('/content') / f'{STEM}_result_preview.mp4'
try:
    PREVIEW.unlink()
except FileNotFoundError:
    pass

subprocess.run([
    'ffmpeg', '-y', '-loglevel', 'error',
    '-i', str(OUTPUT_VIDEO),
    '-vf', 'scale=854:-2',
    '-c:v', 'libx264',
    '-preset', 'veryfast',
    '-crf', '30',
    '-pix_fmt', 'yuv420p',
    '-tag:v', 'avc1',
    '-movflags', '+faststart',
    '-an',
    str(PREVIEW),
], check=True)

print(f'Preview: {PREVIEW} ({PREVIEW.stat().st_size / 1024 / 1024:.1f} MB)')
display(Video(str(PREVIEW), embed=True, width=900, html_attributes='controls'))

print('\nDONE — full result remains on Google Drive:')
print(OUTPUT_VIDEO)
